# Comprehensive EDA for Housing Prices

This notebook walks through a structured exploratory data analysis (EDA) workflow on the California Housing dataset, a widely used benchmark for predicting median house values. We examine the data, engineer insights, and build baseline predictive models using multivariable linear regression, random forests, and (optionally) XGBoost.

## 1. Notebook Setup
- Import core analytical libraries (NumPy, pandas, seaborn, matplotlib).
- Configure plotting aesthetics and display options.
- Fix a random seed for reproducibility.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
sns.set_theme(style='ticks', context='notebook')
pd.set_option('display.max_columns', 40)

### Optional Dependency
Install XGBoost if you plan to run the gradient boosted model comparison.

```
%pip install xgboost
```


In [2]:
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not found. Install it with `pip install xgboost` to run the gradient boosted model section.')

XGBoost not found. Install it with `pip install xgboost` to run the gradient boosted model section.


## 2. Data Loading & Context
The California Housing dataset captures socio-economic and geographic features aggregated at the block group level from the 1990 U.S. Census. The target variable `MedHouseVal` represents median house value in units of $100,000.

Steps:
1. Load the dataset as a pandas `DataFrame`.
2. Separate features (`X`) and target (`y`).
3. Attach metadata such as feature descriptions for reference.

In [6]:
housing

{'data':        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
 0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
 1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
 2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
 3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
 4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
 ...       ...       ...       ...        ...         ...       ...       ...   
 20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
 20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
 20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
 20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
 20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981     39.37   
 
        Longitude 

In [10]:
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target
feature_names = housing.feature_names
feature_desc = housing.DESCR.split('\n')

print(f'Dataset shape: features={X.shape}, target={y.shape}')
X.head()

Dataset shape: features=(20640, 8), target=(20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [11]:
print('Feature documentation (excerpt):')
for line in feature_desc:
    print(line)

Feature documentation (excerpt):
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 

## 3. High-Level Data Overview
Key EDA questions at this stage:
- What columns and dtypes are present?
- Are there missing values or duplicates?
- What are the basic descriptive statistics?

In [12]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [13]:
missing_counts = X.isna().sum().sort_values(ascending=False)
missing_counts

MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64

In [14]:
print(f'Duplicate rows: {X.duplicated().sum()}')

Duplicate rows: 0


In [15]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.870671,1.899822,0.499900,2.563400,3.534800,4.743250,15.000100
HouseAge,20640.0,28.639486,12.585558,1.000000,18.000000,29.000000,37.000000,52.000000
AveRooms,20640.0,5.429000,2.474173,0.846154,4.440716,5.229129,6.052381,141.909091
AveBedrms,20640.0,1.096675,0.473911,0.333333,1.006079,1.048780,1.099526,34.066667
Population,20640.0,1425.476744,1132.462122,3.000000,787.000000,1166.000000,1725.000000,35682.000000
AveOccup,20640.0,3.070655,10.386050,0.692308,2.429741,2.818116,3.282261,1243.333333
Latitude,20640.0,35.631861,2.135952,32.540000,33.930000,34.260000,37.710000,41.950000
Longitude,20640.0,-119.569704,2.003532,-124.350000,-121.800000,-118.490000,-118.010000,-114.310000


## 4. Univariate Exploration
- Visualize the distribution of the target variable.
- Inspect selected feature distributions to understand scale, skew, and potential transformations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(y, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Target distribution: Median House Value')
axes[0].set_xlabel('MedHouseVal (hundreds of thousands USD)')

sns.boxplot(x=y, ax=axes[1], color='coral')
axes[1].set_title('Target distribution (boxplot)')
axes[1].set_xlabel('MedHouseVal')
plt.tight_layout()
plt.show()

In [ ]:
numeric_features = feature_names
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.ravel()
for ax, col in zip(axes, numeric_features):
    sns.histplot(X[col], ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. Bivariate Relationships
- Examine how each feature relates to the target.
- Check correlations among numeric features to detect multicollinearity and guide feature engineering.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.ravel()
for ax, col in zip(axes, numeric_features):
    sns.regplot(x=X[col], y=y, ax=ax, scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
    ax.set_title(f'{col} vs MedHouseVal')
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = X.join(y.rename('MedHouseVal')).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation matrix (features + target)')
plt.show()

## 6. Feature Engineering & Train/Test Split
The California Housing dataset is entirely numeric and has no missing values, so the preprocessing pipeline focuses on scaling for linear models. We keep the tree-based models unscaled via a column transformer.

Steps:
1. Split into training and test sets (80/20).
2. Configure preprocessing pipelines for linear and tree-based algorithms.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
preprocessor_linear = ColumnTransformer(
    transformers=[('num', numeric_transformer, numeric_features)]
)
preprocessor_tree = ColumnTransformer(
    transformers=[('num', 'passthrough', numeric_features)]
)

## 7. Modeling Baselines
We benchmark multiple algorithms under a consistent evaluation protocol (5-fold cross-validation + test set):
- **Multivariable Linear Regression**: captures linear relationships with scaled features.
- **Random Forest Regressor**: handles nonlinear interactions and heterogeneous feature scales.
- **(Optional) XGBoost Regressor**: gradient-boosted trees for potentially stronger performance.

Evaluation metrics:
- Root Mean Squared Error (RMSE)
- Coefficient of Determination ($R^2$) on the hold-out test set.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}
trained_models = {}
model_predictions = {}

LinearPipeline = Pipeline(steps=[
    ('preprocess', preprocessor_linear),
    ('regressor', LinearRegression())
])

RandomForestPipeline = Pipeline(steps=[
    ('preprocess', preprocessor_tree),
    ('regressor', RandomForestRegressor(
        n_estimators=400,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

models = {
    'Linear Regression': LinearPipeline,
    'Random Forest': RandomForestPipeline
}

if XGB_AVAILABLE:
    XGBPipeline = Pipeline(steps=[
        ('preprocess', preprocessor_tree),
        ('regressor', XGBRegressor(
            n_estimators=600,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='reg:squarederror',
            random_state=42
        ))
    ])
    models['XGBoost'] = XGBPipeline

In [ ]:
def evaluate_model(name, pipeline):
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv, scoring='neg_root_mean_squared_error'
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    rmse_test = mean_squared_error(y_test, y_pred, squared=False)
    r2_test = r2_score(y_test, y_pred)

    results[name] = {
        'CV RMSE (mean)': -cv_scores.mean(),
        'CV RMSE (std)': cv_scores.std(),
        'Test RMSE': rmse_test,
        'Test R2': r2_test
    }
    trained_models[name] = pipeline
    model_predictions[name] = y_pred

for model_name, pipeline in models.items():
    evaluate_model(model_name, pipeline)

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Test RMSE')
results_df

## 8. Residual Diagnostics
Residual plots help reveal patterns the model fails to capture (e.g., heteroscedasticity, nonlinearity). We inspect residuals for the linear model and the best-performing tree model.

In [ ]:
linear_preds = model_predictions['Linear Regression']
linear_residuals = y_test - linear_preds
plt.figure(figsize=(7, 4))
sns.scatterplot(x=linear_preds, y=linear_residuals, alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.title('Linear regression residuals vs predictions')
plt.xlabel('Predicted MedHouseVal')
plt.ylabel('Residual')
plt.show()

In [ ]:
best_tree_name = 'Random Forest' if 'Random Forest' in model_predictions else next(iter(model_predictions.keys()))
if best_tree_name != 'Linear Regression':
    tree_preds = model_predictions[best_tree_name]
    tree_residuals = y_test - tree_preds
    plt.figure(figsize=(7, 4))
    sns.scatterplot(x=tree_preds, y=tree_residuals, alpha=0.4, color='forestgreen')
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f'{best_tree_name} residuals vs predictions')
    plt.xlabel('Predicted MedHouseVal')
    plt.ylabel('Residual')
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.kdeplot(linear_residuals, fill=True, label='Linear Regression', ax=ax)
if best_tree_name != 'Linear Regression':
    sns.kdeplot(tree_residuals, fill=True, label=best_tree_name, ax=ax)
ax.set_title('Residual density comparison')
ax.set_xlabel('Residual')
ax.legend()
plt.show()

## 9. Feature Importance Snapshot
Tree-based models offer insights into feature importance. We plot feature importances for the best-performing ensemble model.

In [ ]:
if best_tree_name != 'Linear Regression':
    tree_model = trained_models[best_tree_name]
    regressor = tree_model.named_steps['regressor']
    importances = getattr(regressor, 'feature_importances_', None)
    if importances is not None:
        importance_df = pd.DataFrame({
            'feature': numeric_features,
            'importance': importances
        }).sort_values('importance', ascending=False)
        plt.figure(figsize=(8, 4))
        sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
        plt.title(f'{best_tree_name} feature importance')
        plt.xlabel('Mean decrease in impurity')
        plt.ylabel('Feature')
        plt.show()
    else:
        print(f'Feature importance not available for {best_tree_name}.')
else:
    print('Feature importance visualization skipped (linear regression).')

## 10. Conclusions & Next Steps
- Linear regression provides a fast, interpretable baseline but may underfit nonlinear patterns.
- Random forests (and gradient boosting, when available) typically reduce RMSE by capturing interactions and nonlinearities.
- Residual diagnostics highlight where models struggle—consider feature transformations (e.g., log of median income) or spatial features.
- Future enhancements: hyperparameter tuning, SHAP values for explainability, or incorporating external geographic data to boost accuracy.